# Chapter 4 Notebook

## Setup

In [0]:
%pip install -r requirements.txt

In [0]:
dbutils.library.restartPython()

## Databricks LangChain Components

In [0]:
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    endpoint="databricks-gpt-oss-120b",
    temperature=0,
    max_tokens=256,
)

#invoke the chat model
print(chat_model.invoke("How do I book flights with Unity Airways?"))

In [0]:
from databricks_langchain import VectorSearchRetrieverTool

index_name = "workspace.unity_airways.faq_index"
endpoint_name = "vs_endpoint"

vs_tool = VectorSearchRetrieverTool(
  index_name=index_name,
  tool_name="databricks_docs_retriever",
  tool_description="Retrieves information on frequently asked questions about Unity Airways’ policies, rules, and services",
  num_results=1
)

results = vs_tool.invoke("How do I book flights with Unity Airways?")
print(results)

## Building Tool Calling Agent

### LLM-Only

In [0]:
import mlflow

mlflow.langchain.autolog()

# Set the active model context
logged_model_name = "llm_only"
active_model_info = mlflow.set_active_model(name=logged_model_name)

print(
    f"Active LoggedModel: '{active_model_info.name}', Model ID: '{active_model_info.model_id}'"
)

In [0]:
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    endpoint="databricks-gpt-oss-120b",
    temperature=0,
    max_tokens=256
)

query = "How do I book flights with Unity Airways?"
print(chat_model.invoke(query).content)

### Tool Calling Agent

In [0]:
import mlflow
mlflow.langchain.autolog()

In [0]:
# Set the active model context
logged_model_name = "tool_calling_agent"
active_model_info = mlflow.set_active_model(name=logged_model_name)

print(
    f"Active LoggedModel: '{active_model_info.name}', Model ID: '{active_model_info.model_id}'"
)

In [0]:
import yaml

with open("../conf/chapter04_conf.yml", "r") as f:
    model_config = yaml.safe_load(f)

databricks_resources = model_config.get("databricks_resources")
retriever_config = model_config.get("retriever_tool")
llm_config = model_config.get("llm_config")

In [0]:
from databricks_langchain import ChatDatabricks

model = ChatDatabricks(
    endpoint=databricks_resources.get("model_name"),
    **llm_config.get("llm_parameters")
)

In [0]:
from databricks_langchain import VectorSearchRetrieverTool

vector_search_tool = VectorSearchRetrieverTool(
    index_name=retriever_config.get("index_name"),
    num_results=retriever_config.get('num_results'),
    tool_name=retriever_config.get("tool_name"),
    tool_description=retriever_config.get("tool_description"),
    columns=retriever_config.get("columns")
)

In [0]:
uc_model_conf_path = "../conf/uc_model_registry.yml"
with open(uc_model_conf_path, "r") as f:
    uc_model_conf = yaml.safe_load(f)

In [0]:
system_prompt = model_config.get("system_prompt")

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

# Register a new version of the prompt
prompt_name = uc_model_conf.get("tool_calling_agent_prompt").get("full_name")
prompt = mlflow.genai.register_prompt(
    name=prompt_name,
    template=system_prompt,
    commit_message="Register agent system prompt",
)

print(f"Registered: {prompt.name} (version {prompt.version})")

In [0]:
alias = "champion"

mlflow.genai.set_prompt_alias(
    name=uc_model_conf.get("tool_calling_agent_prompt").get("full_name"),
    alias=alias,
    version=prompt.version,
)

print(f"Alias '{alias}' set to version {prompt.version}")
print(f"\nLoad in code with:")
print(f'  mlflow.genai.load_prompt("prompts:/{prompt_name}@{alias}")')

In [0]:
from langchain.agents import create_agent

promp_template = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@{alias}").template

lc_agent = create_agent(
    model=model,
    tools=[vector_search_tool],
    system_prompt=promp_template,
)

In [0]:
from PIL import Image
from io import BytesIO
from IPython.display import display as image_display

image_byte = lc_agent.get_graph().draw_mermaid_png()
image = Image.open(BytesIO(image_byte))
image_display(image)

image.save("lc_agent.png", "PNG")

In [0]:
lc_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "How do I book flights with Unity Airways?"}
        ]
    }
)

In [0]:
app_params = {
    "model_name": model_config.get("databricks_resources").get("model_name"),
    "retriever_config": model_config.get("retriever_tool"),
    "llm_config": model_config.get("llm_config").get("llm_parameters"),
}
mlflow.log_model_params(model_id=active_model_info.model_id, params=app_params)

## Logging Agent in MLflow

In [0]:
import os
from mlflow.models import infer_signature
from mlflow.models.resources import (DatabricksServingEndpoint, DatabricksVectorSearchIndex)

In [0]:
# Set the active model context
logged_model_name = "tool_calling_agent_with_error"
active_model_info = mlflow.set_active_model(name=logged_model_name)

print(
    f"Active LoggedModel: '{active_model_info.name}', Model ID: '{active_model_info.model_id}'"
)

In [0]:
agent_conf_path = "../conf/chapter04_conf.yml"

with open(agent_conf_path, "r") as f:
    model_config = yaml.safe_load(f)

In [0]:
lc_agent_signature = infer_signature(
    model_input=model_config.get("input_example"),
    model_output=model_config.get("output_example"),
)

lc_agent_signature

In [0]:
from mlflow.models.resources import (DatabricksServingEndpoint, DatabricksVectorSearchIndex)

dependent_resources = [
  DatabricksServingEndpoint(endpoint_name=databricks_resources.get("model_name")),DatabricksVectorSearchIndex(index_name=retriever_config.get("index_name"))
  ]

In [0]:
from mlflow.exceptions import MlflowException
import traceback

# Try to log the agent and capture the specific error
try:
    image_filename = "lc_agent.png"
    
    with mlflow.start_run():
        logged_agent_info = mlflow.langchain.log_model(
            lc_model=lc_agent,
            input_example=model_config.get("input_example"),
            signature=lc_agent_signature,
            resources=dependent_resources,
            pip_requirements="requirements.txt",
            name=active_model_info.name,
            model_id=active_model_info.model_id
        )
        
        mlflow.log_image(mlflow.Image(os.path.join(os.getcwd(), image_filename)), image_filename)
        
except MlflowException as e:
    error_code = e.error_code if hasattr(e, 'error_code') else 'N/A'
    print(f"""
{'='*80}
CAPTURED MLFLOW EXCEPTION:
{'='*80}
Error Type: {type(e).__name__}
Error Code: {error_code}

Error Message:
{str(e)}
{'='*80}
Full Traceback:
{'='*80}""")
    traceback.print_exc()
    
except Exception as e:
    print(f"""
{'='*80}
CAPTURED GENERAL EXCEPTION:
{'='*80}
Error Type: {type(e).__name__}

Error Message:
{str(e)}
{'='*80}
Full Traceback:
{'='*80}""")
    traceback.print_exc()

## Models from Code

### Create new tool_calling_agent.py script

In [0]:
%%writefile tool_calling_agent.py

import os
import yaml
import mlflow

from operator import itemgetter

from databricks_langchain import ChatDatabricks
from databricks_langchain import VectorSearchRetrieverTool

from langchain.agents import create_agent

## Enable MLflow Tracing
mlflow.langchain.autolog()

try:
    with open("chapter04_conf.yml", "r") as f:
        model_config = yaml.safe_load(f)

    with open("uc_model_registry.yml", "r") as f:
        uc_model_conf = yaml.safe_load(f)

except:
    with open("../conf/chapter04_conf.yml", "r") as f:
        model_config = yaml.safe_load(f)

    with open("../conf/uc_model_registry.yml", "r") as f:
        uc_model_conf = yaml.safe_load(f)

databricks_resources = model_config.get("databricks_resources")
retriever_config = model_config.get("retriever_tool")
llm_config = model_config.get("llm_config")

# Define LLM
model = ChatDatabricks(
    endpoint=databricks_resources.get("model_name"),
    **llm_config.get("llm_parameters")
)

# Define VS Tool
vector_search_tool = VectorSearchRetrieverTool(
    index_name=retriever_config.get("index_name"),
    num_results=retriever_config.get('num_results'),
    tool_name=retriever_config.get("tool_name"),
    tool_description=retriever_config.get("tool_description"),
    columns=retriever_config.get("columns")
)

# Load System Prompt
prompt_name = uc_model_conf.get("tool_calling_agent_prompt").get("full_name")
promp_template = mlflow.genai.load_prompt(f"prompts:/{prompt_name}@champion").template
system_prompt = model_config.get("system_prompt")

# Create Agent
lc_agent = create_agent(
    model=model,
    tools=[vector_search_tool],
    system_prompt=system_prompt,
)

## Set Model for Models from Code Logging to Work
mlflow.models.set_model(model=lc_agent)

In [0]:
dbutils.library.restartPython()

In [0]:
import yaml

with open("../conf/chapter04_conf.yml", "r") as f:
    model_config = yaml.safe_load(f)

databricks_resources = model_config.get("databricks_resources")
retriever_config = model_config.get("retriever_tool")
llm_config = model_config.get("llm_config")

from tool_calling_agent import lc_agent

### Log to MLflow

In [0]:
import mlflow 

# Set the active model context
logged_model_name = "tool_calling_agent_from_code"
active_model_info = mlflow.set_active_model(name=logged_model_name)

print(
    f"Active LoggedModel: '{active_model_info.name}', Model ID: '{active_model_info.model_id}'"
)

In [0]:
lc_agent.invoke(model_config.get("input_example"))

In [0]:
import os
from mlflow.models import infer_signature
from mlflow.models.resources import (DatabricksServingEndpoint, DatabricksVectorSearchIndex)

lc_agent_signature = infer_signature(
    model_input=model_config.get("input_example"),
    model_output=model_config.get("output_example"),
)

dependent_resources = [
  DatabricksServingEndpoint(endpoint_name=databricks_resources.get("model_name")),
  DatabricksVectorSearchIndex(index_name=retriever_config.get("index_name"))
  ]

In [0]:
image_filename = "lc_agent.png"
tool_calling_agent_script_path = "tool_calling_agent.py"
tool_calling_agent_conf_path = "../conf/chapter04_conf.yml"

uc_model_conf_path = "../conf/uc_model_registry.yml"
with open(uc_model_conf_path, "r") as f:
    uc_model_conf = yaml.safe_load(f)

with mlflow.start_run():

    logged_agent_info = mlflow.langchain.log_model(
        lc_model=tool_calling_agent_script_path,
        input_example=model_config.get("input_example"),
        signature=lc_agent_signature,
        resources=dependent_resources,
        pip_requirements="requirements.txt",
        code_paths=[tool_calling_agent_conf_path, uc_model_conf_path],
        name=active_model_info.name,
        model_id=active_model_info.model_id,
        registered_model_name=uc_model_conf.get("tool_calling_agent").get("full_name"),
    )

    mlflow.log_image(
        mlflow.Image(os.path.join(os.getcwd(), image_filename)), image_filename
    )


In [0]:
# Test the agent locally
loaded_agent = mlflow.langchain.load_model(logged_agent_info.model_uri)
loaded_agent.invoke(model_config.get("input_example"))